# Sprint 4 — RAG com página, seção e estado operacional

**Challenge FIAP — Forzy | Processamento de Linguagem Natural**

Este notebook compara o preset legado com uma recuperação em português que preserva a origem de cada trecho e usa alertas ativos para orientar a busca.

| Seção | Conteúdo |
|---|---|
| 0 | Objetivo e roteiro |
| 1 | Setup local e chave do OpenRouter |
| 2 | Extração por página, chunking e comparação dos presets |
| 3 | Embeddings, reranker e indexação |
| 4 | Estado operacional a partir das leituras reais |
| 5 | Busca com e sem estado |
| 6 | Conclusões e limites |
| 7 | Assistente de troubleshooting |
| 8 | Smoke do assistente |
| 9 | Gabarito de avaliação (20 dentro do escopo + 5 fora) |
| 10 | Recuperação: LEGADO × NOVO sem estado × NOVO com estado |
| 11 | Juiz por afirmação: protocolo e calibração |
| 12 | Respostas: faithfulness, relevancy e abstenção |
| 13 | Análise de erros |


## 1. Setup

Execute a partir da raiz do repositório. O bloco localiza a raiz sem clonar o projeto nem instalar pacotes; as dependências ficam em `requirements.txt`. A chave é lida do `.env` da raiz e nunca exibida.


In [ ]:
from pathlib import Path
import sys

candidatos_raiz = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
ROOT = next((candidato for candidato in candidatos_raiz if (candidato / "src" / "pln" / "config.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório que contém src/pln/config.py.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pln.config import NOVO, LEGADO, PDF_CATALOGO_FILE
from src.pln.llm import load_openrouter_api_key

try:
    chave_openrouter_disponivel = bool(load_openrouter_api_key(ROOT / ".env"))
except RuntimeError:
    chave_openrouter_disponivel = False

print(f"Raiz do repositório: {ROOT}")
print(f"PDF do catálogo encontrado: {PDF_CATALOGO_FILE.is_file()}")
print(f"Chave OpenRouter disponível: {chave_openrouter_disponivel}")


## 2. Extração e chunking

A extração remove os elementos repetidos do cabeçalho/rodapé, mantém o número impresso, identifica seções e converte tabelas em linhas rotuladas. O preset `LEGADO` mantém o agrupamento por parágrafo sem metadados; `NOVO` mantém página e seção.


In [ ]:
from src.pln.config import LEGADO, NOVO, PDF_CATALOGO_FILE
from src.pln.rag import chunk_legado, chunk_novo, extrair_paginas

paginas = extrair_paginas(PDF_CATALOGO_FILE)
chunks_legado = chunk_legado("\n\n".join(pagina.texto_original for pagina in paginas), LEGADO)
chunks_novos = chunk_novo(paginas, NOVO)

print(f"Páginas extraídas: {len(paginas)}")
print(f"Chunks LEGADO: {len(chunks_legado)} | limite: {LEGADO.chunk_max} caracteres")
print(f"Chunks NOVO: {len(chunks_novos)} | limite: {NOVO.chunk_max} caracteres")
print(f"Chunks NOVO com página e seção: {sum(chunk.pagina is not None and bool(chunk.secao) for chunk in chunks_novos)}")
for chunk in chunks_novos[:3]:
    print(f"\nPágina {chunk.pagina} — seção {chunk.secao} — {chunk.secao_titulo}\n{chunk.texto}")


## 3. Indexação

O índice usa ChromaDB efêmero com distância cosseno. O preset `NOVO` aplica embeddings E5 multilíngues e reranking multilíngue. O tempo abaixo inclui o carregamento dos modelos na primeira execução e a criação do índice.


In [ ]:
from time import perf_counter

from src.pln.rag import criar_retriever

inicio_indexacao = perf_counter()
retriever = criar_retriever(chunks_novos, NOVO)
tempo_indexacao = perf_counter() - inicio_indexacao
print(f"Embedding: {NOVO.modelo_embedding}")
print(f"Reranker: {NOVO.modelo_reranker}")
print(f"Passagens indexadas: {len(chunks_novos)}")
print(f"Tempo de carregamento e indexação: {tempo_indexacao:.1f} s")


## 4. Estado operacional

Os episódios são derivados das leituras reais de sensores. A primeira amostra usa o início de um alerta crítico; a segunda usa uma leitura real fora de qualquer janela de alerta.


In [ ]:
from src.pln.alertas import carregar_leituras, detectar_episodios
from src.pln.estado import construir_estado_operacional

leituras = carregar_leituras()
alertas = detectar_episodios(leituras)
alerta_critico = next(alerta for alerta in alertas if alerta.severidade == "critico")
instante_critico = alerta_critico.inicio
instantes_leitura = sorted((leitura["timestamp"] for leitura in leituras), reverse=True)
instante_normal = next(
    instante
    for instante in instantes_leitura
    if not any(alerta.inicio <= instante <= alerta.fim for alerta in alertas)
)
estado_critico = construir_estado_operacional(alertas, instante_critico)
estado_normal = construir_estado_operacional(alertas, instante_normal)

print(f"Episódios derivados: {len(alertas)}")
print(f"Instante crítico: {estado_critico.texto()}")
print(f"Instante normal: {estado_normal.texto()}")


## 5. Retriever com e sem estado

A mesma pergunta percorre o retriever em duas condições. Sem estado, não há expansão por termos operacionais nem bônus de seção. Com estado, a busca inclui os termos do domínio ativo e pode receber o bônus definido no preset.


In [ ]:
pergunta = "Quais verificações devo fazer para entender o aquecimento do motor?"

resultados_sem_estado = retriever.buscar(pergunta, estado=None)
resultados_com_estado = retriever.buscar(pergunta, estado=estado_critico)

def mostrar_resultados(rotulo, resultados):
    print(f"\n{rotulo}")
    for posicao, resultado in enumerate(resultados, start=1):
        print(
            f"{posicao}. p. {resultado['pagina']} — seção {resultado['secao']} — "
            f"{resultado['secao_titulo']} | denso={resultado['score_denso']:.3f} "
            f"rerank={resultado['score_rerank']:.3f} final={resultado['score_final']:.3f}"
        )
        print(resultado["texto"][:320])

mostrar_resultados("Sem estado", resultados_sem_estado)
mostrar_resultados("Com estado crítico", resultados_com_estado)


## 6. Conclusão

O preset `NOVO` mantém página e seção nos chunks, serializa tabelas em linhas legíveis e expande a busca com termos associados aos alertas ativos. `LEGADO` permanece disponível para a comparação planejada.

A seção 5 permite observar a mudança de consulta e de ordenação com o mesmo enunciado. O resultado depende dos modelos multilíngues configurados e do catálogo W22; não deve ser interpretado como avaliação de qualidade sem o conjunto de perguntas e métricas da próxima etapa.

**Execução local:** rode as células em ordem para baixar/carregar os modelos e verificar os resultados reais. Não salve outputs ou chaves no notebook.


## 7. Assistente de troubleshooting

O assistente usa o estado operacional como dado confiável e os chunks recuperados como única fonte técnica. O score bruto do reranker determina abstenção e confiança; página e seção dos chunks validam as citações. A memória mantém até `MEMORIA_TURNOS` turnos e amplia consultas curtas ou anafóricas com a pergunta anterior.

### Inspeção offline e ponto de partida do limiar

Inspecionei 10 consultas dentro do escopo — incluindo duas buscas já compostas com a pergunta anterior — e 6 consultas fora dele. Os valores abaixo são `score_rerank_bruto` do primeiro chunk retornado pelo retriever, usando o mesmo estado crítico passado ao smoke. A regra de partida é somar a margem mínima ao maior score fora do escopo e arredondar o resultado para cima ao décimo: `ceil((maior_score_fora + margem) * 10) / 10`.

O maior score fora do escopo foi -3.027231. Com margem mínima de 0.1, o valor intermediário é -2.927231 e o arredondamento para cima resulta em -2.9, o ponto de partida desta inspeção. Nesta amostra pequena nenhum limiar separa as duas populações: o menor score dentro do escopo (-4.751304, seguimento de ventilação) é menor que o maior score fora dele (-3.027231). O valor vigente de `LIMIAR_ABSTENCAO` (2.0) não vem desta inspeção: foi calibrado no conjunto `dev` do gabarito, na seção 10.

| Escopo | Consulta usada na medição | Melhor score bruto | Página/seção do primeiro chunk |
|---|---|---:|---|
| Dentro | Qual é o grau de proteção do motor WEG W22? | 9.925054 | 18 / 7 |
| Dentro | Qual é o grau de proteção do motor WEG W22? E o que essa proteção significa? | 8.729171 | 18 / 7 |
| Dentro | Qual é a classe de isolamento do motor WEG W22? | 6.704764 | 20 / 11.1 |
| Dentro | Quais são as características do sistema de ventilação? | -4.239105 | 21 / 12.1 |
| Dentro | Quais são as características do sistema de ventilação? E quais são as características desse sistema? | -4.751304 | 21 / 12.1 |
| Dentro | Quais dados técnicos aparecem na placa de identificação do motor WEG W22? | 9.336778 | 12 / 3.11 |
| Dentro | Qual é a faixa de temperatura ambiente permitida para o motor WEG W22? | 7.954395 | 19 / 9 |
| Dentro | Qual altitude de instalação é permitida para o motor WEG W22? | 6.590040 | 20 / 11.1 |
| Dentro | Como deve ser feita a lubrificação dos rolamentos do motor WEG W22? | 4.264139 | 14 / 5.1 |
| Dentro | Quais são os limites de vibração permitidos para o motor WEG W22? | 7.368129 | 14 / 5.2 |
| Fora | Como assar pão de queijo? | -6.147628 | 20 / 11.1 |
| Fora | Qual é a população atual do Brasil? | -3.027231 | 38 / 16 |
| Fora | Como configurar uma impressora a laser? | -6.134845 | 33 / 15 |
| Fora | Qual é a capital da França? | -3.982897 | 38 / 16 |
| Fora | Como preparar um bolo de chocolate? | -6.812472 | 38 / 16 |
| Fora | Como configurar um roteador Wi-Fi doméstico? | -5.741543 | 33 / 15 |


In [ ]:
from IPython.display import Markdown, display
from src.pln.config import (
    FAIXAS_CONFIANCA,
    LIMIAR_ABSTENCAO,
    MARGEM_ABSTENCAO,
    LIMIAR_PALAVRAS_CONSULTA,
    MEMORIA_TURNOS,
    MODELO_GERADOR,
    PROMPT_SISTEMA_ASSISTENTE,
)

display(Markdown(f"```text\n{PROMPT_SISTEMA_ASSISTENTE}\n```"))
display(Markdown(
    "|Parâmetro|Valor|\n|---|---|\n"
    f"|Modelo gerador|`{MODELO_GERADOR}`|\n"
    f"|Memória em turnos|`{MEMORIA_TURNOS}`|\n"
    f"|Limiar de consulta curta|`{LIMIAR_PALAVRAS_CONSULTA}`|\n"
    f"|Limiar de abstenção|`{LIMIAR_ABSTENCAO}`|\n"
    f"|Margem mínima do limiar|`{MARGEM_ABSTENCAO}`|\n"
    f"|Faixas de confiança|`{FAIXAS_CONFIANCA}`|"
))


## 8. Smoke do assistente

Execute as três perguntas dentro do escopo, os dois seguimentos de memória e as três perguntas fora do escopo. Cada resultado mostra a consulta usada, fontes, confiança, abstenção e modelo. Para respostas não abstidas, o smoke também imprime o texto completo, incluindo citações e a linha de confiança. Esta seção faz chamadas ao gerador e depende da chave local do OpenRouter.


In [ ]:
from src.pln.assistente import Assistente

perguntas_smoke = (
    "Qual é o grau de proteção do motor WEG W22?",
    "E o que essa proteção significa?",
    "Qual é a classe de isolamento do motor WEG W22?",
    "Quais são as características do sistema de ventilação?",
    "E quais são as características desse sistema?",
    "Como assar pão de queijo?",
    "Qual é a população atual do Brasil?",
    "Como configurar uma impressora a laser?",
)
assistente_demo = Assistente(retriever)
respostas_smoke = []
for pergunta_smoke in perguntas_smoke:
    resposta_smoke = assistente_demo.responder(pergunta_smoke, estado=estado_critico)
    respostas_smoke.append(resposta_smoke)
    print(
        {
            "pergunta": pergunta_smoke,
            "consulta_usada": resposta_smoke.consulta_usada,
            "fontes": resposta_smoke.fontes,
            "confianca": resposta_smoke.confianca,
            "abstencao": resposta_smoke.abstencao,
            "modelo": resposta_smoke.modelo,
        }
    )
    if not resposta_smoke.abstencao:
        print(resposta_smoke.texto)


## 9. Gabarito de avaliação

O gabarito (`data/avaliacao_rag.json`) tem 20 perguntas **dentro do escopo**, cada uma com resposta de referência, página(s) e seção(ões) do PDF e um `trecho_ancora`, frase literal do catálogo que sustenta a resposta; e 5 perguntas **fora do escopo**, cuja resposta correta é a abstenção. Cinco das 20 formam o `dev` (usado para calibrar parâmetros) e 15 formam o `teste`. Nove perguntas dentro do escopo trazem um cenário de estado operacional, construído a partir de alertas reais: o estado no início do primeiro alerta crítico que envolve só aquela grandeza. Duas perguntas fora do escopo também recebem estado, para conferir que ele não derruba a abstenção.

O teste da âncora abaixo falha se alguma frase do gabarito não existir na página indicada, e se o termo-chave de uma pergunta fora do escopo existir no PDF.

In [ ]:
from collections import Counter

from src.pln.avaliacao import ancora_presente, carregar_gabarito, construir_estados, termo_ausente_no_pdf, textos_por_pagina

perguntas = carregar_gabarito()
dentro = [pergunta for pergunta in perguntas if pergunta.escopo == "dentro"]
fora = [pergunta for pergunta in perguntas if pergunta.escopo == "fora"]
print(f"Dentro do escopo: {len(dentro)} | fora do escopo: {len(fora)}")
print("Split (dentro):", dict(Counter(pergunta.split for pergunta in dentro)))
print("Categorias (dentro):", dict(Counter(pergunta.categoria for pergunta in dentro)))
estados_usados = Counter(pergunta.estado for pergunta in dentro if pergunta.estado)
print("Perguntas dentro do escopo com estado:", sum(estados_usados.values()), dict(estados_usados))

for identificador in ("e02", "m05", "f01"):
    exemplo = next(pergunta for pergunta in perguntas if pergunta.id == identificador)
    print(f"\n[{exemplo.id}] {exemplo.pergunta}")
    if exemplo.escopo == "dentro":
        print(f"  referência: {exemplo.resposta_referencia}")
        print(f"  páginas {list(exemplo.paginas_gabarito)} | seções {list(exemplo.secoes_gabarito)} | {exemplo.categoria} | {exemplo.split}")
        print(f"  âncora: \"{exemplo.trecho_ancora}\"")
    else:
        print(f"  esperado: {exemplo.esperado} (o termo \"{exemplo.termo_ausente}\" não ocorre no PDF)")

textos_pdf = textos_por_pagina(paginas)
sem_ancora = [pergunta.id for pergunta in dentro if not ancora_presente(pergunta, textos_pdf)]
termo_presente = [pergunta.id for pergunta in fora if not termo_ausente_no_pdf(pergunta, textos_pdf)]
print(f"\nPerguntas cuja âncora não está na página do gabarito: {sem_ancora}")
print(f"Perguntas fora do escopo cujo termo-chave existe no PDF: {termo_presente}")
assert not sem_ancora and not termo_presente

estados = construir_estados(alertas)
for chave, estado_cenario in estados.items():
    print(f"\n{chave}: {estado_cenario.texto()}")

## 10. Recuperação

Um chunk é **relevante por página** quando o intervalo de páginas dele toca uma página do gabarito, e **relevante de forma estrita** quando contém o `trecho_ancora`. Os chunks do `LEGADO` não trazem metadados: para medi-los, cada chunk recebe as páginas em que seus parágrafos aparecem no PDF.

Para cada pergunta dentro do escopo, `hit@k` diz se há algum chunk relevante entre os `k` primeiros; `precision@k` é a fração de relevantes entre os `k`; `MRR@k` é o inverso da posição do primeiro relevante; `context precision@k` é a precisão média nas posições dos relevantes (peso pela posição). `k` ∈ {1, 4}.

O grupo `com_estado` reúne só as perguntas que têm cenário de estado: é onde `NOVO sem estado` e `NOVO com estado` realmente diferem, porque nas demais as duas configurações recebem a mesma consulta.

### Calibração de `PESO_ESTADO` e `LIMIAR_ABSTENCAO` (somente `dev`)

A grade abaixo usa só as 5 perguntas `dev` dentro do escopo e as 5 fora do escopo. A escolha maximiza o hit@4 entre os pares que se abstêm em todas as perguntas fora do escopo; o desempate prefere menos abstenções indevidas nas perguntas dentro do escopo. O `teste` só é lido depois, nas tabelas seguintes.

In [ ]:
import pandas as pd
from IPython.display import display

from src.pln.avaliacao import escolher_parametros, varrer_parametros
from src.pln.config import LIMIAR_ABSTENCAO, PESO_ESTADO

grade_dev = varrer_parametros(retriever, perguntas, estados)
display(pd.DataFrame(grade_dev))
escolha = escolher_parametros(grade_dev)
print(f"Par escolhido no dev: PESO_ESTADO={escolha['peso_estado']} | LIMIAR_ABSTENCAO={escolha['limiar_abstencao']}")
print(f"Par em config.py:     PESO_ESTADO={PESO_ESTADO} | LIMIAR_ABSTENCAO={LIMIAR_ABSTENCAO}")
assert (escolha["peso_estado"], escolha["limiar_abstencao"]) == (PESO_ESTADO, LIMIAR_ABSTENCAO), "config.py diverge da calibração"

In [ ]:
from src.pln.avaliacao import ConfiguracaoRecuperacao, executar_recuperacao, resumir_recuperacao
from src.pln.config import KS_AVALIACAO
from src.pln.rag import criar_retriever

retriever_legado = criar_retriever(chunks_legado, LEGADO)
configuracoes = [
    ConfiguracaoRecuperacao("LEGADO", retriever_legado, None, paginas),
    ConfiguracaoRecuperacao("NOVO sem estado", retriever, None),
    ConfiguracaoRecuperacao("NOVO com estado", retriever, estados),
]
registros_recuperacao = {c.nome: executar_recuperacao(c, perguntas, KS_AVALIACAO) for c in configuracoes}
resumos_recuperacao = {nome: resumir_recuperacao(regs, KS_AVALIACAO) for nome, regs in registros_recuperacao.items()}
tabela_recuperacao = pd.DataFrame(
    [{"configuracao": nome, **linha} for nome, linhas in resumos_recuperacao.items() for linha in linhas]
)
COLUNAS = ["configuracao", "grupo", "k", "n", "hit", "precision", "mrr", "context_precision"]


def mostrar_recuperacao(criterio, grupos):
    recorte = tabela_recuperacao[(tabela_recuperacao.criterio == criterio) & tabela_recuperacao.grupo.isin(grupos)]
    display(recorte.sort_values(["grupo", "k", "configuracao"])[COLUNAS].round(3).reset_index(drop=True))


print("Relevância por página — teste (15), dev (5) e perguntas com estado")
mostrar_recuperacao("pagina", ["teste", "dev", "com_estado"])
print("Relevância estrita (chunk contém a âncora) — teste, dev e perguntas com estado")
mostrar_recuperacao("estrita", ["teste", "dev", "com_estado"])
print("Relevância por página — teste, por categoria")
mostrar_recuperacao("pagina", [f"teste/{categoria}" for categoria in ("elétrica", "mecânica", "preventiva", "especificação")])

In [ ]:
def valor(nome, grupo, criterio, k, metrica):
    linha = tabela_recuperacao[
        (tabela_recuperacao.configuracao == nome) & (tabela_recuperacao.grupo == grupo)
        & (tabela_recuperacao.criterio == criterio) & (tabela_recuperacao.k == k)
    ]
    return float(linha[metrica].iloc[0])


def comparar(melhor, base, grupo):
    for criterio in ("pagina", "estrita"):
        for metrica in ("hit", "mrr", "context_precision"):
            a, b = valor(melhor, grupo, criterio, 4, metrica), valor(base, grupo, criterio, 4, metrica)
            sentido = "supera" if a > b else "empata com" if a == b else "NÃO supera"
            print(f"[{grupo}/{criterio}] {metrica}@4: {melhor} {a:.3f} {sentido} {base} {b:.3f}")


comparar("NOVO sem estado", "LEGADO", "teste")
print()
comparar("NOVO com estado", "NOVO sem estado", "com_estado")

In [ ]:
from src.pln.avaliacao import gravar_json
from src.pln.config import AVALIACAO_RECUPERACAO_OUTPUT_FILE

gravar_json(
    AVALIACAO_RECUPERACAO_OUTPUT_FILE,
    {
        "ks": list(KS_AVALIACAO),
        "parametros": {"peso_estado": PESO_ESTADO, "limiar_abstencao": LIMIAR_ABSTENCAO},
        "grade_dev": grade_dev,
        "configuracoes": {
            nome: {"resumo": resumos_recuperacao[nome], "registros": registros_recuperacao[nome]}
            for nome in registros_recuperacao
        },
    },
)
print(f"Gravado: {AVALIACAO_RECUPERACAO_OUTPUT_FILE.name}")

## 11. Juiz por afirmação

A resposta é dividida em frases por código (sem LLM), ignorando a linha `Confiança:`, as citações `[p., seç.]` e a frase de abstenção. O juiz (`MODELO_JUIZ`, de família diferente do gerador, temperatura 0, com cache) recebe os chunks usados e uma afirmação por vez e devolve **um** rótulo:

| Rótulo | Significado |
|---|---|
| `SUSTENTADA` | decorre do contexto |
| `EXTERNA` | pode até ser verdadeira, mas **não está no contexto** |
| `CONTRADIZ` | contraria o contexto |
| `META` | não é afirmação factual (fora da conta) |

**faithfulness = SUSTENTADA / (SUSTENTADA + EXTERNA + CONTRADIZ)**. Duas guardas determinísticas prevalecem sobre o juiz: qualquer número da afirmação ausente do contexto, ou um marcador de conhecimento externo (`conhecimento geral`, `em geral`, `normalmente`… lista em `config.py`; ignorado se o próprio contexto usa a expressão), torna a afirmação `EXTERNA`. Resposta de abstenção sem conteúdo factual tem faithfulness 1,0, e a abstenção é avaliada à parte.

### Calibração

`data/juiz_calibracao.json` traz 16 casos `(contexto, afirmação, rótulo esperado)` escritos à mão: 5 `EXTERNA` (incluindo o `V-ring` e a "classe de isolamento F" quando o contexto não a traz), 4 `CONTRADIZ`, 5 `SUSTENTADA` e 2 `META`. Seis casos formam o `dev` da calibração; o prompt só pode ser ajustado (até 3 vezes) olhando esses seis. A versão do prompt abaixo é a primeira: acertou os 6 casos `dev`, então nenhum ajuste foi feito, e os 10 casos `teste` só foram lidos nesta execução. O resultado é reportado como saiu, nos dois modos: o juiz sozinho e o juiz com as guardas.

In [ ]:
from IPython.display import Markdown

from src.pln.avaliacao import Juiz, calibrar_juiz, carregar_casos_calibracao
from src.pln.config import JUIZ_CALIBRACAO_OUTPUT_FILE, MODELO_GERADOR, MODELO_JUIZ, PROMPT_JUIZ_AFIRMACAO
from src.pln.llm import create_openrouter_client

assert MODELO_JUIZ != MODELO_GERADOR
cliente_llm = create_openrouter_client()
juiz = Juiz(cliente_llm)
display(Markdown(f"Modelo do juiz: `{MODELO_JUIZ}` | modelo do gerador: `{MODELO_GERADOR}`\n\n```text\n{PROMPT_JUIZ_AFIRMACAO}\n```"))

calibracao = calibrar_juiz(juiz, carregar_casos_calibracao())


def formatar(valor):
    return "n/d" if valor is None else f"{valor:.3f}"


for split in ("dev", "teste", "todos"):
    for modo in ("juiz_isolado", "juiz_com_guardas"):
        metricas = calibracao["metricas"][split][modo]
        print(f"{split:6} {modo:17} n={metricas['n']:2} acurácia={formatar(metricas['acuracia'])} "
              f"revocação EXTERNA={formatar(metricas['revocacao_externa'])} precisão EXTERNA={formatar(metricas['precisao_externa'])}")

print("\nMatriz de confusão (todos os casos; linhas = esperado, colunas = previsto)")
display(pd.DataFrame(calibracao["metricas"]["todos"]["juiz_isolado"]["matriz_confusao"]).T.rename_axis("juiz isolado"))
display(pd.DataFrame(calibracao["metricas"]["todos"]["juiz_com_guardas"]["matriz_confusao"]).T.rename_axis("juiz com guardas"))

divergentes = [caso for caso in calibracao["casos"] if caso["final"] != caso["esperado"]]
print(f"\nCasos em que o resultado final diverge do rótulo esperado: {len(divergentes)}")
for caso in divergentes:
    print(f"  {caso['id']} ({caso['split']}): esperado {caso['esperado']}, juiz {caso['juiz']}, final {caso['final']} — {caso['afirmacao']}")

gravar_json(JUIZ_CALIBRACAO_OUTPUT_FILE, {**calibracao, "prompt": PROMPT_JUIZ_AFIRMACAO, "ajustes_de_prompt_com_dev": 0})

## 12. Respostas

O assistente responde as 25 perguntas com `MODELO_GERADOR`, na configuração `NOVO com estado` (a pergunta recebe o cenário de estado, quando tem um). A memória é reiniciada a cada pergunta, para que uma não herde a outra. Para cada resposta, o juiz rotula as afirmações (faithfulness) e compara a resposta com a referência (answer relevancy: `COBRE` = 1, `PARCIAL` = 0,5, `NAO_COBRE` = 0, mais o cosseno entre os embeddings de resposta e referência como segundo sinal).

Duas regras de abstenção: abstenção em pergunta **dentro** do escopo vale `NAO_COBRE`; resposta factual a pergunta **fora** do escopo é `alucinacao_fora_escopo`, nunca acerto. Falha persistente de API marca a pergunta como `erro`, sem trocar de modelo. Os `data/saida/avaliacao_*.json` guardam, por pergunta, modelo, consulta, chunks (página, seção, scores), resposta, afirmações rotuladas e métricas.

In [ ]:
from src.pln.assistente import Assistente
from src.pln.avaliacao import ContextoRespostas, executar_respostas, resumir_respostas
from src.pln.config import AVALIACAO_RESPOSTAS_OUTPUT_FILE

assistente_avaliacao = Assistente(retriever, cliente_llm)
contexto_respostas = ContextoRespostas(assistente_avaliacao, juiz, retriever.componentes.embedding, estados)
registros_respostas = executar_respostas(contexto_respostas, perguntas)
resumo_respostas = resumir_respostas(registros_respostas)

linhas_respostas = [
    {
        "id": r["id"], "escopo": r["escopo"], "split": r["split"], "abstencao": r.get("abstencao"),
        "faithfulness": r.get("faithfulness"), "afirmações": r.get("contagem"),
        "cobertura": r.get("cobertura"), "cosseno": r.get("cosseno"),
        "alucinação": r.get("alucinacao_fora_escopo"), "erro": r.get("erro"),
    }
    for r in registros_respostas
]
display(pd.DataFrame(linhas_respostas).round(3))
display(pd.DataFrame({k: v for k, v in resumo_respostas.items() if k != "por_categoria"}).round(3))
display(pd.DataFrame(resumo_respostas["por_categoria"]).T.round(3))

In [ ]:
geral = resumo_respostas["geral"]
print("Entrega anterior: faithfulness 4,75/5 e relevancy 3,50/5, nota 1–5 de um juiz único sobre a resposta inteira, sem context precision.")
print("Agora: faithfulness = fração de afirmações sustentadas (0–1) e relevancy em {0; 0,5; 1}, por afirmação e contra referência.")
print("O método mudou: os números não são comparáveis em escala; a conversão abaixo (nota/5) só dá ordem de grandeza.")
print(f"  anterior: faithfulness {4.75 / 5:.2f} | relevancy {3.50 / 5:.2f}")
print(f"  agora:    faithfulness {geral['faithfulness_media']:.2f} | relevancy {geral['relevancy_media']:.2f}")
print(f"Abstenções indevidas (dentro do escopo): {geral['abstencoes_indevidas']} | corretas (fora): {geral['abstencoes_corretas']}"
      f" | alucinações fora do escopo: {geral['alucinacoes_fora_escopo']} | erros: {geral['erros']}")

gravar_json(
    AVALIACAO_RESPOSTAS_OUTPUT_FILE,
    {"modelo_gerador": MODELO_GERADOR, "modelo_juiz": MODELO_JUIZ, "resumo": resumo_respostas, "registros": registros_respostas},
)
print(f"Gravado: {AVALIACAO_RESPOSTAS_OUTPUT_FILE.name}")

## 13. Análise de erros

As cinco piores respostas pela média entre faithfulness e pontos de relevancy (pergunta fora do escopo vale 1 se houve abstenção e 0 se houve resposta factual). A **causa provável** é uma regra mecânica, não um julgamento: *recuperação* quando nenhum dos 4 chunks cai numa página do gabarito (ou o assistente se absteve); *geração* quando a recuperação acertou mas há afirmações `EXTERNA` ou `CONTRADIZ`; *gabarito ou juiz* quando a recuperação acertou, nada foi marcado como externo e ainda assim a cobertura não é total — caso para conferir a `resposta_referencia` e a resposta lado a lado.

In [ ]:
recuperacao_por_id = {r["id"]: r for r in registros_recuperacao["NOVO com estado"]}
por_id = {pergunta.id: pergunta for pergunta in perguntas}


def nota(registro):
    if registro.get("erro"):
        return 0.0
    if registro["escopo"] == "fora":
        return 1.0 if registro["abstencao_correta"] else 0.0
    fidelidade = registro.get("faithfulness")
    return ((1.0 if fidelidade is None else fidelidade) + registro["pontos"]) / 2


def causa_provavel(registro):
    if registro.get("erro"):
        return "erro de API"
    if registro["escopo"] == "fora":
        return "geração (respondeu fora do escopo)"
    achou_pagina = recuperacao_por_id[registro["id"]]["metricas"]["pagina"]["4"]["hit"] == 1.0
    contagem = registro.get("contagem", {})
    if registro["abstencao"] or not achou_pagina:
        return "recuperação"
    if contagem.get("EXTERNA", 0) or contagem.get("CONTRADIZ", 0):
        return "geração"
    return "gabarito ou juiz"


piores = sorted(registros_respostas, key=nota)[:5]
for registro in piores:
    pergunta = por_id[registro["id"]]
    paginas_recuperadas = [chunk["pagina"] for chunk in registro.get("chunks", [])]
    print(f"[{registro['id']}] nota {nota(registro):.2f} | causa provável: {causa_provavel(registro)}")
    print(f"  pergunta: {pergunta.pergunta}")
    print(f"  páginas do gabarito {list(pergunta.paginas_gabarito)} × recuperadas {paginas_recuperadas}")
    print(f"  resposta: {registro.get('resposta', registro.get('mensagem_erro'))}")
    for afirmacao in registro.get("afirmacoes", []):
        print(f"    - {afirmacao['rotulo']:10} {afirmacao['afirmacao']}")
    print()

**Execução local:** rode as células em ordem; as seções 11 a 13 chamam o OpenRouter com a chave do `.env` da raiz. Não salve a chave no notebook.